<div style="background:linear-gradient(135deg,#1A5276 0%,#2E86C1 100%);padding:40px 36px 32px 36px;border-radius:10px;margin-bottom:8px;">
  <p style="color:#AED6F1;font-size:13px;margin:0 0 6px 0;letter-spacing:2px;">CURSO 8 · MÓDULO 2 · CLASE 5</p>
  <h1 style="color:white;font-size:36px;margin:0 0 10px 0;font-weight:700;">Ejercicios: Regresión Logística desde el GLM</h1>
  <p style="color:#AED6F1;font-size:16px;margin:0 0 24px 0;font-style:italic;">Logit, odds, sigmoide e interpretación de coeficientes</p>
  <hr style="border-color:#5DADE2;margin:0 0 20px 0;">
  <p style="color:#EBF5FB;font-size:13px;margin:0;">📌 <strong>Docente:</strong> Josef Rodriguez &nbsp;·&nbsp; 🟢 Básico · 🟡 Intermedio · 🔴 Avanzado</p>
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
np.set_printoptions(precision=4,suppress=True)
plt.rcParams.update({'figure.dpi':110,'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
SEED=42; np.random.seed(SEED)
print('✅ Setup listo')

---
## 🟢 Ejercicio 1 — Logit, odds y sigmoide

1. Calcular logit(p) para p ∈ {0.1, 0.25, 0.5, 0.75, 0.9}
2. Calcular los odds correspondientes
3. Verificar que sigmoide(logit(p)) = p
4. Graficar las tres funciones en una sola figura

In [ ]:
p_vals = np.array([0.1, 0.25, 0.5, 0.75, 0.9])
# --- Tu código aquí ---

In [ ]:
# ✅ SOLUCIÓN
p_vals=np.array([0.1,0.25,0.5,0.75,0.9])
odds=p_vals/(1-p_vals)
logit=np.log(odds)
recuperado=expit(logit)
df_res=pd.DataFrame({'p':p_vals,'odds':odds.round(4),'logit':logit.round(4),'σ(logit)':recuperado.round(4)})
print(df_res.to_string(index=False))
print(f'\nσ(logit(p))=p: {np.allclose(recuperado,p_vals)}')
fig,axes=plt.subplots(1,3,figsize=(13,4))
pv=np.linspace(0.001,0.999,400); zv=np.linspace(-6,6,400)
axes[0].plot(pv,pv/(1-pv),color='#2E86C1',lw=2); axes[0].set(xlabel='p',ylabel='odds',title='Odds',ylim=(0,10)); axes[0].grid(True,alpha=0.2)
axes[1].plot(pv,np.log(pv/(1-pv)),color='#8E44AD',lw=2); axes[1].set(xlabel='p',ylabel='logit(p)',title='Logit'); axes[1].grid(True,alpha=0.2)
axes[2].plot(zv,expit(zv),color='#27AE60',lw=2); axes[2].axhline(0.5,color='gray',lw=0.8,linestyle='--'); axes[2].set(xlabel='z',ylabel='σ(z)',title='Sigmoide'); axes[2].grid(True,alpha=0.2)
plt.tight_layout(); plt.show()

---
## 🟢 Ejercicio 2 — Interpretar coeficientes

Un modelo de abandono de clientes da estos coeficientes:
- β₀ = -2.1
- β_edad = -0.04 (edad en años)
- β_recencia = 0.08 (días sin comprar)
- β_ticket = -0.003 (ticket promedio en USD)

1. ¿Qué odds ratio corresponde a cada coeficiente?
2. ¿Cuál es la probabilidad de abandono de un cliente de 35 años, 90 días sin comprar, ticket=500?
3. ¿Qué variable tiene mayor impacto en los odds?
4. ¿Aumentar el ticket en 100 USD multiplica los odds por cuánto?

In [ ]:
beta_churn = np.array([-2.1, -0.04, 0.08, -0.003])
feat_names = ['intercepto','edad','recencia','ticket']
# --- Tu código aquí ---

In [ ]:
# ✅ SOLUCIÓN
beta_churn=np.array([-2.1,-0.04,0.08,-0.003])
feat_names=['intercepto','edad','recencia','ticket']
print('Odds Ratios:')
for nm,b in zip(feat_names,beta_churn):
    OR=np.exp(b)
    print(f'  {nm:12s}: β={b:+.4f}  OR=e^β={OR:.4f}  {"→ aumenta riesgo" if b>0 else "→ reduce riesgo"}')
client=[1,35,90,500]; logit_c=np.dot(beta_churn,client)
p_churn=expit(logit_c)
print(f'\nCliente (35 años, 90 días, $500): logit={logit_c:.4f} → P(abandono)={p_churn:.4f}')
print(f'Impacto ticket +100 USD: OR={np.exp(-0.003*100):.4f} → multiplica odds por {np.exp(-0.003*100):.4f}')

---
## 🟡 Ejercicio 3 — Log-verosimilitud desde cero

1. Implementar `log_lik(beta, X, y)` sin usar sklearn
2. Implementar el gradiente `grad_log_lik(beta, X, y)`
3. Maximizar con scipy.optimize.minimize
4. Comparar con sklearn — ¿coinciden los coeficientes?

In [ ]:
np.random.seed(SEED)
n_ej3=300; X_raw=np.random.randn(n_ej3,3)
X_ej3=np.column_stack([np.ones(n_ej3),X_raw])
beta_real3=np.array([-0.5,1.5,-1.0,0.8])
y_ej3=np.random.binomial(1,expit(X_ej3@beta_real3))
# --- Tu código aquí ---

In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED)
n_ej3=300; X_raw=np.random.randn(n_ej3,3)
X_ej3=np.column_stack([np.ones(n_ej3),X_raw])
beta_real3=np.array([-0.5,1.5,-1.0,0.8])
y_ej3=np.random.binomial(1,expit(X_ej3@beta_real3))

def log_lik(b,X,y):
    p=np.clip(expit(X@b),1e-10,1-1e-10)
    return np.sum(y*np.log(p)+(1-y)*np.log(1-p))

def neg_ll(b,X,y): return -log_lik(b,X,y)/len(y)
def grad_nll(b,X,y): return -X.T@(y-expit(X@b))/len(y)

res=minimize(neg_ll,np.zeros(4),args=(X_ej3,y_ej3),method='BFGS',jac=grad_nll)
beta_mle=res.x

lr3=LogisticRegression(C=1e6,solver='lbfgs',max_iter=500)
Xtr3,Xte3,ytr3,yte3=train_test_split(X_ej3[:,1:],y_ej3,test_size=0.2,random_state=SEED)
lr3.fit(Xtr3,ytr3)

print(f'β_real:   {beta_real3}')
print(f'β_mle:    {beta_mle.round(4)}')
print(f'β_sklearn:[{lr3.intercept_[0]:.4f}] + {lr3.coef_[0].round(4)}')
print(f'\nConvergió: {res.success}')
print(f'Acc test:  {accuracy_score(yte3,lr3.predict(Xte3)):.4f}')

---
## 🔴 Ejercicio 4 — Pipeline completo de scoring

Implementá `logistic_pipeline(X_raw, y, feats)` que:
1. Estandarice X, agregue y no agregue intercepto (sklearn lo maneja)
2. Estime el modelo
3. Imprima tabla de coeficientes con OR
4. Evalúe accuracy y reporte de clasificación
5. Prediga para 3 nuevos clientes dados

In [ ]:
np.random.seed(SEED); n_s=1000
edad_s=np.random.randint(20,70,n_s).astype(float)
ingr_s=np.random.normal(45000,18000,n_s)
deuda_s=np.random.uniform(0.1,0.9,n_s)
logit_s=-2+0.03*edad_s-0.00002*ingr_s+2.5*deuda_s
y_s=np.random.binomial(1,expit(logit_s))
X_s_raw=np.column_stack([edad_s,ingr_s,deuda_s])
feats_s=['edad','ingresos','deuda_ratio']
nuevos_s=np.array([[30,60000,0.2],[55,25000,0.8],[40,45000,0.5]])
# --- Tu código aquí ---

In [ ]:
# ✅ SOLUCIÓN
def logistic_pipeline(X_raw,y,feats,nuevos=None):
    sc=StandardScaler(); Xsc=sc.fit_transform(X_raw)
    Xtr,Xte,ytr,yte=train_test_split(Xsc,y,test_size=0.2,random_state=SEED)
    lr=LogisticRegression(C=1e6,solver='lbfgs',max_iter=500); lr.fit(Xtr,ytr)
    print(f'\n{"Feature":15s} {"β̂":>8s} {"OR":>10s}')
    print('─'*36)
    for nm,b in zip(['intercepto']+feats,np.concatenate([lr.intercept_,lr.coef_[0]])):
        print(f'  {nm:13s} {b:>8.4f} {np.exp(b):>10.4f}')
    acc=accuracy_score(yte,lr.predict(Xte))
    print(f'\nAccuracy: {acc:.4f}')
    print(classification_report(yte,lr.predict(Xte),target_names=['No','Sí']))
    if nuevos is not None:
        nuevos_sc=sc.transform(nuevos)
        p_n=lr.predict_proba(nuevos_sc)[:,1]
        print('Nuevos clientes:')
        for i,p in enumerate(p_n):
            dec='✓ BAJO RIESGO' if p<0.3 else ('⚠ REVISAR' if p<0.5 else '✗ ALTO RIESGO')
            print(f'  Cliente {i+1}: P(evento)={p:.3f}  {dec}')

logistic_pipeline(X_s_raw,y_s,feats_s,nuevos_s)

---
<div style="background:#1A5276;color:white;padding:20px 24px;border-radius:8px;">
<strong>Próxima clase</strong><br>Deviance · AIC · Wald test · Pseudo R² · Curvas ROC/AUC<br>
<em>Docente: Josef Rodriguez · Curso 8 · Módulo 2</em>
</div>